# 00 — 벤치마크 전처리 (EDA + samples.jsonl 작성)

새 벤치마크 1개를 다음 구조로 만드는 **템플릿**.

```
BENCHMARK/<bench_id>/
├── audio/*.wav      ← 16kHz mono PCM_16
└── samples.jsonl    ← 메타 + 정규화 텍스트 (한 줄 = 한 발화)
```

## 사용법

1. 본 노트북을 새 이름으로 복사 (예: `00_build_<corpus>.ipynb`)
2. **§2 의 변수 5개만** 수정
3. Run All — 위에서 아래로 셀 실행
4. 마지막 셀의 검증 통과하면 끝
5. 그 다음 `BENCHMARK/Benchmark_Meta_Table.xlsx` 의 `BENCHMARKS` 시트에 행 한 줄 추가

## 흐름

| 섹션 | 내용 |
|---|---|
| §2 | 변수 설정 (수정 영역) |
| §3 | 오디오 EDA (SR / 채널 / 길이 분포) |
| §4 | 메타 EDA (NaN / 분포 / 이중 전사) |
| §5 | 16kHz mono 변환 |
| §6 | samples.jsonl 작성 |
| §7 | 검증 |


## §2. 변수 설정 (수정 영역)

다음 5개 변수만 수정:

| 변수 | 의미 |
|---|---|
| `CORPUS_ID` | `data/Corpora_Meta_Table.xlsx` 의 `Title (U)` 와 일치 |
| `BENCH_ID` | `<corpus>_<속성>_<조건>` 패턴. BENCHMARK 폴더명 = JSONL key prefix |
| `SRC_AUDIO` | 원본 wav 폴더 (보통 `/data/ASR-BENCHMARK/RAW/<source>/audio/`) |
| `SRC_META` | 원본 메타 CSV/JSONL 경로 |
| `REPO_ROOT` | 본 저장소 클론 절대경로 |


In [ ]:
from pathlib import Path
import sys

# ── 수정 영역 ───────────────────────────────────────────────────
CORPUS_ID  = "AIHub_NewCorpus"                                            # Corpora_Meta_Table 의 Title (U)
BENCH_ID   = f"{CORPUS_ID}_general_clean"                                  # <corpus>_<속성>_<조건>
SRC_AUDIO  = Path("/data/ASR-BENCHMARK/RAW/<source>/audio")                # 원본 wav 폴더
SRC_META   = Path("/data/ASR-BENCHMARK/RAW/<source>/metadata.csv")         # 원본 메타 CSV
REPO_ROOT  = Path("/home/cssong/workspace/TRAIN-ASR")                      # 클론 경로
# ───────────────────────────────────────────────────────────────

sys.path.insert(0, str(REPO_ROOT))

# 산출 경로 (자동)
BENCH_DIR  = REPO_ROOT / "BENCHMARK" / BENCH_ID
AUDIO_DIR  = BENCH_DIR / "audio"
JSONL_PATH = BENCH_DIR / "samples.jsonl"

print(f"CORPUS_ID  : {CORPUS_ID}")
print(f"BENCH_ID   : {BENCH_ID}")
print(f"SRC_AUDIO  : {SRC_AUDIO}  (exists={SRC_AUDIO.exists()})")
print(f"SRC_META   : {SRC_META}   (exists={SRC_META.exists()})")
print(f"→ AUDIO_DIR : {AUDIO_DIR}")
print(f"→ JSONL_PATH: {JSONL_PATH}")


## §3. 오디오 EDA

샘플레이트 / 채널 / 인코딩 / 발화 길이 분포 / 클리핑 비율 점검.

판정 기준:
- `sample_rate`: 16000 아니면 §5 변환 필요
- `p95 > 30s`: 모델 입력 한계 초과 발화 있음 → 평가셋에서 제외 검토
- `clip ratio > 0.05`: 5% 이상 클리핑 → 음원 품질 문제 가능


In [ ]:
import numpy as np
import soundfile as sf
from collections import Counter

files = sorted(SRC_AUDIO.glob("*.wav"))
print(f"파일 수: {len(files)}\n")

sr_dist, ch_dist, subtype = Counter(), Counter(), Counter()
durations, peaks = [], []

for p in files:
    info = sf.info(p)
    sr_dist[info.samplerate] += 1
    ch_dist[info.channels] += 1
    subtype[info.subtype] += 1
    durations.append(info.duration)
    # 클리핑 검사 — 표본 200건만 (전수 검사는 큰 파일 부담)
    if len(peaks) < 200:
        audio, _ = sf.read(p, dtype="float32")
        peaks.append(float(np.abs(audio).max()))

durations = np.array(durations)
print(f"sample_rate: {dict(sr_dist)}")
print(f"channels:    {dict(ch_dist)}")
print(f"subtype:     {dict(subtype)}")
print(
    f"duration(s): mean={durations.mean():.2f}  median={np.median(durations):.2f}  "
    f"p5={np.percentile(durations,5):.2f}  p95={np.percentile(durations,95):.2f}  "
    f"max={durations.max():.2f}"
)
print(f"clip ratio:  {sum(1 for p in peaks if p >= 0.99) / max(len(peaks),1):.3f}")


## §4. 메타 EDA

행 수 / NaN 비율 / 컬럼별 분포 / 전사 길이 / 이중 전사·잡음 태그 비율.

판정 기준:
- 전사 NaN > 0: 빈 발화 — JSONL 작성 시 제외 필요
- 메타 NaN > 30%: 슬라이스 평가 약화 (예: `age_group` 이 30% 비면 연령대별 분석 의미 ↓)
- 이중 전사 `(A)/(B)` > 0: `normalize_korean_asr` 가 *철자(앞)* 자동 선택
- `speaker_id` 없음: 화자 누수 검증 불가 → 임시 ID 부여 (`unknown_<corpus>_<idx>`)

> 메타 파일의 컬럼명이 `wav_id` / `utterance` / `speaker_id` / `age_group` / `gender` 가 아니면 아래 코드에서 컬럼명 변경.


In [ ]:
import pandas as pd

df = pd.read_csv(SRC_META, encoding="utf-8-sig")
print(f"rows: {len(df)}")
print(f"columns: {df.columns.tolist()}\n")

print("[NaN 비율]")
print(df.isna().mean().round(3))

print("\n[메타 분포]")
for col in ("age_group", "gender", "domain", "dialect"):
    if col in df.columns:
        print(f"  {col}: {df[col].value_counts(dropna=False).to_dict()}")

print("\n[화자]")
if "speaker_id" in df.columns:
    spk = df["speaker_id"].value_counts()
    print(
        f"  unique: {len(spk)}  /  평균 발화수: {spk.mean():.1f}  /  "
        f"max: {spk.max()} (top {spk.head(3).to_dict()})"
    )
else:
    print("  speaker_id 컬럼 없음 — 임시 ID 부여됨 (§6)")

# 전사 — 컬럼명이 다르면 'utterance' 부분 변경
print("\n[전사]")
texts = df["utterance"].astype(str)
char_len = texts.str.len()
word_len = texts.str.split().str.len()
print(
    f"  문자 수: mean={char_len.mean():.1f}  median={char_len.median():.0f}  "
    f"p95={char_len.quantile(0.95):.0f}  max={char_len.max()}"
)
print(f"  어절 수: mean={word_len.mean():.1f}  median={word_len.median():.0f}")

print("\n[특수 패턴]")
print(f"  중복 발화:        {texts.duplicated().mean():.3f}")
print(f"  이중 전사 (A)/(B): {texts.str.contains(r'\\([^)]+\\)/\\([^)]+\\)', regex=True).mean():.3f}")
print(f"  영문 포함:        {texts.str.contains(r'[A-Za-z]', regex=True).mean():.3f}")
print(f"  숫자 포함:        {texts.str.contains(r'[0-9]', regex=True).mean():.3f}")
print(f"  한자 포함:        {texts.str.contains(r'[\u4e00-\u9fff]', regex=True).mean():.3f}")


## §5. 16kHz mono PCM_16 변환

원본 wav → `BENCHMARK/<bench_id>/audio/`.

이미 변환된 파일은 skip (중복 변환 방지). 처음 실행 시 시간 소요 큼 (파일 수 비례).


In [ ]:
import librosa
import soundfile as sf

AUDIO_DIR.mkdir(parents=True, exist_ok=True)

n_converted, n_skipped = 0, 0
for src in SRC_AUDIO.glob("*.wav"):
    dst = AUDIO_DIR / src.name
    if dst.exists():
        n_skipped += 1
        continue
    audio, _ = librosa.load(src, sr=16000, mono=True)
    sf.write(dst, audio, 16000, subtype="PCM_16")
    n_converted += 1

total = len(list(AUDIO_DIR.glob("*.wav")))
print(f"converted: {n_converted}  /  skipped: {n_skipped}  /  total in {AUDIO_DIR.name}/: {total}")


## §6. samples.jsonl 작성

메타 CSV → `Sample` 객체 → `samples.jsonl` (한 줄 = 한 발화).

필수 필드 10개 — `key / audio / duration_sec / text / text_normalized / corpus_id / speaker_id / age_group / gender / labeling`.

`text_normalized` 는 `normalize_korean_asr(text)` 결과 (정규화 함수 한 곳 — `project/evaluation/normalize.py`).

> 메타 파일의 컬럼명이 다르면 `row['wav_id']` / `row['utterance']` 부분 변경.


In [ ]:
import csv
import soundfile as sf
from project.data.schema import Sample
from project.data import save_samples
from project.evaluation import normalize_korean_asr

samples = []
n_missing = 0

with SRC_META.open(encoding="utf-8-sig") as f:
    for i, row in enumerate(csv.DictReader(f)):
        wav = AUDIO_DIR / f"{row['wav_id']}.wav"   # wav_id 컬럼명 다르면 변경
        if not wav.exists():
            n_missing += 1
            continue
        info = sf.info(wav)
        text = row["utterance"].strip()             # 텍스트 컬럼명 다르면 변경
        if not text:
            continue

        samples.append(Sample(
            key            = f"{BENCH_ID}_{i:06d}",
            audio          = str(wav.resolve()),    # 절대경로 (schema 검증 요구)
            duration_sec   = round(info.duration, 3),
            text           = text,
            text_normalized= normalize_korean_asr(text),
            corpus_id      = CORPUS_ID,
            speaker_id     = row.get("speaker_id") or f"unknown_{CORPUS_ID}_{i:06d}",
            age_group      = row.get("age_group", "unknown"),
            gender         = row.get("gender", "unknown"),
            labeling       = "human-review",        # 평가셋은 human-review 필수
        ))

n = save_samples(samples, JSONL_PATH)
print(f"wrote {n} samples → {JSONL_PATH}")
if n_missing:
    print(f"⚠️ wav 누락 {n_missing} 건 (메타에는 있으나 audio/ 에 없음)")


## §7. 검증

스키마 검증 — `audio` 절대경로 / 필수 필드 / `labeling == "human-review"` / `corpus_id` 메타시트 일치.

에러 시 형식: `[line N] <필드> <사유>` — 해당 필드만 수정 후 재실행.

화자 누수 검증은 학습셋(`data/GOLD/train.jsonl`)이 있을 때 추가로 호출.


In [ ]:
from project.data import load_samples, validate_samples

loaded = load_samples(JSONL_PATH)
validate_samples(loaded, for_test=True)
print(f"✓ 검증 통과 ({len(loaded)} samples)")

# 첫 줄 미리보기
import json
print("\n[첫 줄 미리보기]")
print(json.dumps(loaded[0].to_dict(), ensure_ascii=False, indent=2))


## §8. 다음 단계 — 메타시트 행 추가 + 평가 실행

### 메타시트 (수동)

`BENCHMARK/Benchmark_Meta_Table.xlsx` 의 `BENCHMARKS` 시트 맨 아래 행 추가:

| 컬럼 | 값 |
|---|---|
| `D` | `o` |
| `Benchmark ID` | `<BENCH_ID>` (이 노트북 §2 의 값) |
| `Source Corpus` | `<CORPUS_ID>` (Corpora_Meta_Table 의 Title (U) 와 일치) |
| `# Utts` / `Hours` / `# Speakers` / `Avg Dur (s)` | §3·§4 EDA 결과 |
| 그 외 (Domain / Speaker Group / Condition 등) | 같은 파일의 **INFO 시트** 카탈로그 참고 |

### 평가 실행

`BENCHMARK/configs/<name>.yaml` 의 `benchmarks` 리스트에 `<BENCH_ID>` 추가 → 평가 실행 (BENCHMARK/README.md §3 참고).
